In [1]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================
# Paths
# ======================
BASE_PATH = "/kaggle/input/datasets/ashishjangra27/face-mask-12k-images-dataset/Face Mask Dataset"

train_path = os.path.join(BASE_PATH, "Train")
test_path = os.path.join(BASE_PATH, "Test")

# Function to load data
def load_data(path):
    images = []
    labels = []

    for label_name in ["WithMask", "WithoutMask"]:
        class_path = os.path.join(path, label_name)
        label = 1 if label_name == "WithMask" else 0

        for img_name in os.listdir(class_path):
            img = Image.open(os.path.join(class_path, img_name)).convert("L")
            img = img.resize((64, 64))
            images.append(np.array(img) / 255.0)
            labels.append(label)

    X = np.array(images).reshape(-1, 1, 64, 64)
    y = np.array(labels).reshape(-1, 1)

    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Load train and test sets
X_train, y_train = load_data(train_path)
X_test, y_test = load_data(test_path)

# Move to GPU
X_train, y_train = X_train.to(device), y_train.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

# ======================
# Model
# ======================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Conv2d(1, 16, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(16 * 31 * 31, 1)

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = SimpleCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ======================
# Training
# ======================
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

# ======================
# Evaluation
# ======================
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        preds = torch.sigmoid(outputs) > 0.5
        correct += (preds.float() == yb).sum().item()
        total += yb.size(0)

print("Test Accuracy:", correct / total)

Using device: cuda
Epoch [1/10], Loss: 97.0587
Epoch [2/10], Loss: 46.6023
Epoch [3/10], Loss: 35.4106
Epoch [4/10], Loss: 28.0138
Epoch [5/10], Loss: 24.0209
Epoch [6/10], Loss: 22.0167
Epoch [7/10], Loss: 20.4464
Epoch [8/10], Loss: 18.5459
Epoch [9/10], Loss: 17.4545
Epoch [10/10], Loss: 16.1620
Test Accuracy: 0.9707661290322581
